#### Fusionne les données brutes avec le référentiel Aurehal pour récupérer les coordonnées connues et structurer un fichier de base dédoublonné.

In [1]:
# =========================================================
#  MASTER CONFIGURATION - VARIABLES GLOBALES
# Cette partie permet de spécifier le nom du tableau Excel à analyser et le nom des colonnes
# ===============================================================================================

# --- 1. CHEMINS DES FICHIERS ---
FILE_MAIN    = "copublications_Inria_2018-2024_sans_villes.xlsx"          # Entrée brute
FILE_REF_ID  = "ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx"     # Fichier de référence ID
FILE_CITIES  = "../cities500/cities500.txt"                            # Base Geonames
FILE_INTERIM = "Copublis_Internationales_Inria_nov_2025_enrichi.xlsx"  # Sortie Script 1 / Entrée Script 2
FILE_FINAL   = "copublications_Inria_2018-2024_avec_villes.xlsx"
FILE_FOUND   = "Resultat_Villes_TROUVEES.xlsx"                         # Rapport succès
FILE_MISSING = "Resultat_Villes_MANQUANTES.xlsx"                       # Rapport échecs

# --- 2. COLONNES DU FICHIER PRINCIPAL (Excel Input) ---
COL_MAIN_ID      = "id_Aurehal_org_copubliant"             # Clé de jointure
COL_MAIN_ADDR    = "Adresse_org_Top_copubliant"                # Pour recherche texte
COL_MAIN_COUNTRY = "Nom_Pays_org_copubliant"                   # Pour filtrer les pays
COL_MAIN_ORG     = "Nom_org_Top_copubliant"   # Pour recherche texte (et crochets)

# --- 3. COLONNES DU FICHIER DE RÉFÉRENCE (Excel Ref) & CIBLES ---
# Ces noms seront aussi utilisés comme noms de colonnes dans le fichier final
COL_REF_ID        = "id_Aurehal_org_copubliant" # Clé de jointure dans le fichier Ref
COL_REF_LAT       = "Latitude"   # Colonne Latitude (Existante ou à créer)
COL_REF_LON       = "Longitude"  # Colonne Longitude (Existante ou à créer)
COL_REF_CITY      = "Ville"      # Colonne Ville (Existante ou à créer)
COL_REF_GEOID     = "geonameid"  # Colonne ID Geonames (Existante ou à créer)
COL_REF_STATE_SRC = "StateCode"       # Nom colonne État dans fichier Ref (sera renommée)

# --- 4. COLONNES INTERNES / STANDARDS ---
COL_INTERNAL_STATE = "Statecode" # Nom final pour la colonne État normalisée

# --- 5. COLONNES DU FICHIER GEONAMES (cities500.txt) ---
# Noms à donner aux colonnes lors du chargement du fichier TXT
COL_GEO_ID      = "geonameid"
COL_GEO_NAME    = "name"
COL_GEO_ASCII   = "asciiname"
COL_GEO_LAT     = "lat"
COL_GEO_LON     = "lon"
COL_GEO_COUNTRY = "country_code"

print("✅ Configuration chargée. Toutes les variables sont prêtes.")

✅ Configuration chargée. Toutes les variables sont prêtes.


In [2]:
#==========================================
# Première analyse de l'adresse 
# - ajout des infos déjà présentes dans le dictionnaire
# - recherche des villes encore manquantes (ajout du code de l'Etat pour les US)
# ==========================================================================================
import pandas as pd
import re
import unidecode
from tqdm import tqdm

# Activer tqdm pour pandas
tqdm.pandas()

# Mapping des États US (Nom -> Code)
US_STATES_MAP = {
    "ALABAMA": "AL", "ALASKA": "AK", "ARIZONA": "AZ", "ARKANSAS": "AR", "CALIFORNIA": "CA",
    "COLORADO": "CO", "CONNECTICUT": "CT", "DELAWARE": "DE", "FLORIDA": "FL", "GEORGIA": "GA",
    "HAWAII": "HI", "IDAHO": "ID", "ILLINOIS": "IL", "INDIANA": "IN", "IOWA": "IA",
    "KANSAS": "KS", "KENTUCKY": "KY", "LOUISIANA": "LA", "MAINE": "ME", "MARYLAND": "MD",
    "MASSACHUSETTS": "MA", "MICHIGAN": "MI", "MINNESOTA": "MN", "MISSISSIPPI": "MS", "MISSOURI": "MO",
    "MONTANA": "MT", "NEBRASKA": "NE", "NEVADA": "NV", "NEW HAMPSHIRE": "NH", "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM", "NEW YORK": "NY", "NORTH CAROLINA": "NC", "NORTH DAKOTA": "ND", "OHIO": "OH",
    "OKLAHOMA": "OK", "OREGON": "OR", "PENNSYLVANIA": "PA", "RHODE ISLAND": "RI", "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD", "TENNESSEE": "TN", "TEXAS": "TX", "UTAH": "UT", "VERMONT": "VT",
    "VIRGINIA": "VA", "WASHINGTON": "WA", "WEST VIRGINIA": "WV", "WISCONSIN": "WI", "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC"
}
VALID_US_CODES = set(US_STATES_MAP.values())
STOPWORDS = {"la", "le", "de", "del", "da", "di", "el", "univ", "university", "institute"}

# ---------------------------------------------------------
# 2. FONCTIONS UTILITAIRES
# ---------------------------------------------------------

def normalize_text(text):
    """Nettoyage basique pour comparaison."""
    if pd.isna(text): return ""
    return unidecode.unidecode(str(text)).lower().strip()

def fix_camel_case(text):
    """Insère un espace entre une minuscule et une majuscule."""
    if pd.isna(text): return text
    return re.sub(r'([a-z])([A-Z])', r'\1 \2', str(text))

def extract_us_state(address):
    """Repère un code d'État US ou un nom complet."""
    if pd.isna(address): return None
    clean_addr = unidecode.unidecode(str(address)).replace(",", " ").replace(".", " ").upper()
    tokens = clean_addr.split()
    
    # 1. Code exact (GA, NY) à la fin
    for token in reversed(tokens):
        if token in VALID_US_CODES:
            return token
    # 2. Nom complet
    for state_name, state_code in US_STATES_MAP.items():
        if re.search(r'\b' + re.escape(state_name) + r'\b', clean_addr):
            return state_code
    return None

def extract_and_clean_brackets(row, col_org):
    """Analyse les crochets et nettoie le nom de l'organisme."""
    org_name = row[col_org]
    if pd.isna(org_name): return None, org_name

    m = re.search(r"\[(.*?)\]", str(org_name))
    city_found = None
    org_cleaned = org_name

    if m:
        content = m.group(1).strip()
        norm_content = normalize_text(content)
        if len(norm_content) >= 3 and norm_content not in STOPWORDS:
            city_found = content
            # Supprimer les crochets et nettoyer les espaces
            org_cleaned = re.sub(r"\[.*?\]", "", str(org_name)).strip()
            org_cleaned = re.sub(r"\s+", " ", org_cleaned)

    return city_found, org_cleaned

# ---------------------------------------------------------
# 3. CHARGEMENT GEONAMES
# ---------------------------------------------------------
print("--- Chargement de cities500 ---")
df_cities = pd.read_csv(FILE_CITIES, sep="\t", usecols=[0, 1, 2, 10], names=["geonameid", "name", "asciiname", "admin1_code"], dtype=str)
liste_names = set(df_cities["name"].dropna())

# ---------------------------------------------------------
# 4. PIPELINE PRINCIPAL
# ---------------------------------------------------------
print("--- Chargement des fichiers de données ---")
df_left = pd.read_excel(FILE_MAIN)
df_right = pd.read_excel(FILE_REF_ID)

# --- A) Fusion par ID Aurehal depuis le dictionnaire des infos déjà existantes ---
print("--- Étape 1 : Fusion par ID Aurehal ---")

# On vérifie si 'Etat' (COL_REF_STATE_SRC) existe et on le renomme potentiellement vers 'Statecode'
if COL_REF_STATE_SRC in df_right.columns and COL_INTERNAL_STATE not in df_right.columns:
    df_right = df_right.rename(columns={COL_REF_STATE_SRC: COL_INTERNAL_STATE})

# On définit les colonnes qu'on AIMERAIT avoir (Mélange des config et noms internes)
target_cols = [COL_REF_ID, COL_REF_LAT, COL_REF_LON, COL_REF_GEOID, COL_REF_CITY, COL_INTERNAL_STATE]

# On ne sélectionne QUE celles qui existent vraiment dans le fichier right
existing_cols = [c for c in target_cols if c in df_right.columns]
df_tmp = df_right[existing_cols]

merged = df_left.merge(
    df_tmp,
    how="left",
    left_on=COL_MAIN_ID,     # Utilisation variable config
    right_on=COL_REF_ID,     # Utilisation variable config
    suffixes=("", "_right")
)

# Remplissage des Latitudes, Longitudes, Geonames
# On itère sur les noms de colonnes cibles (Latitude, Longitude, etc.)
cols_to_fill = [COL_REF_LAT, COL_REF_LON, COL_REF_GEOID, COL_REF_CITY, COL_INTERNAL_STATE]

for col in cols_to_fill:
    col_right = f"{col}_right"
    if col in merged.columns and col_right in merged.columns:
        merged[col] = merged[col].fillna(merged[col_right])
    elif col not in merged.columns and col_right in merged.columns:
        # Si la colonne n'existait pas dans le fichier gauche mais existe dans le droit
        merged[col] = merged[col_right]

# Nettoyage des colonnes temporaires
# On supprime tout ce qui finit par _right ou l'ID de droite dupliqué
merged.drop(columns=[c for c in merged.columns if c.endswith("_right") or c == f"{COL_REF_ID}_right"], inplace=True)

# --- B) Normalisation & États US (AVEC FILTRE PAYS) ---
print("--- Étape 2 : Normalisation Adresses & États US ---")

if COL_MAIN_ADDR in merged.columns:
    # 1. Normalisation de l'adresse (Espace entre minuscule et Majuscule)
    merged[COL_MAIN_ADDR] = merged[COL_MAIN_ADDR].progress_apply(fix_camel_case)
    
    # 2. États US : On ne cherche QUE si le pays est United States
    if COL_INTERNAL_STATE not in merged.columns:
        merged[COL_INTERNAL_STATE] = None

    # Création d'un masque : Vrai si le pays ressemble aux USA
    if COL_MAIN_COUNTRY in merged.columns:
        mask_usa = merged[COL_MAIN_COUNTRY].astype(str).str.lower().str.contains(r"united states|usa|etats-unis|u\.s\.a\.", na=False, regex=True)
    else:
        print(f"⚠️ Attention : Colonne '{COL_MAIN_COUNTRY}' introuvable. Risque de faux positifs pour les États US.")
        mask_usa = merged[COL_MAIN_ADDR].notna() # Fallback

    # On combine : Il faut que ce soit aux USA ET que l'adresse ne soit pas vide
    mask_process = mask_usa & merged[COL_MAIN_ADDR].notna()
    
    print(f"-> Recherche des codes États sur {mask_process.sum()} lignes (Filtre USA actif)...")

    # Application de l'extraction UNIQUEMENT sur les lignes filtrées
    merged.loc[mask_process, "Statecode_detected"] = merged.loc[mask_process, COL_MAIN_ADDR].progress_apply(extract_us_state)
    
    # Remplissage de la colonne finale
    merged[COL_INTERNAL_STATE] = merged[COL_INTERNAL_STATE].fillna(merged["Statecode_detected"])
    
    # Nettoyage
    if "Statecode_detected" in merged.columns:
        merged.drop(columns=["Statecode_detected"], inplace=True)

# --- C) Crochets & Nettoyage: récupération de la ville indiquée entre crochets dans le nom de l'organisme ---
print("--- Étape 3 : Analyse des crochets ---")
if COL_MAIN_ORG in merged.columns:
    merged[["City_bracket", "Organisme_clean"]] = merged.apply(
        lambda x: pd.Series(extract_and_clean_brackets(x, COL_MAIN_ORG)), axis=1
    )
    
    if COL_REF_CITY not in merged.columns: merged[COL_REF_CITY] = None
    merged[COL_REF_CITY] = merged[COL_REF_CITY].fillna(merged["City_bracket"])
    merged[COL_MAIN_ORG] = merged["Organisme_clean"] # Mise à jour nom
    merged.drop(columns=["City_bracket", "Organisme_clean"], inplace=True)

# --- D) Recherche manquants ---
print("--- Étape 4 : Recherche Ville dans l'adresse ---")
def find_city_in_addr(addr):
    if pd.isna(addr): return None
    tokens = re.split(r'[,\s]+', str(addr))
    for token in reversed(tokens):
        if len(token) > 2 and token.strip() in liste_names:
            return token.strip()
    return None

mask_still_missing = merged[COL_REF_CITY].isna() & merged[COL_MAIN_ADDR].notna()
print(f"Villes cherchées par adresse : {mask_still_missing.sum()}")
if mask_still_missing.sum() > 0:
    merged.loc[mask_still_missing, COL_REF_CITY] = merged.loc[mask_still_missing, COL_MAIN_ADDR].progress_apply(find_city_in_addr)

# ---------------------------------------------------------
# 5. SAUVEGARDE & ANALYSE FINALE (MISE À JOUR)
# ---------------------------------------------------------
FILE_OUTPUT_GEONAMES_DEUPL = "Copublis_Trouve_Geonames_ID_Dedoublonne.xlsx"
FILE_OUTPUT_MISSING_DEUPL = "Copublis_Manquants_Dedoublonne.xlsx"

# Sauvegarde du fichier principal enrichi
merged.to_excel(FILE_INTERIM, index=False)
print(f"✅ Terminé : {FILE_INTERIM}")

# --- Préparation des Données pour le Dé-doublonnage ---
ORG_COL_USED = COL_MAIN_ORG
if ORG_COL_USED not in merged.columns and "Organisme_clean" in merged.columns:
    ORG_COL_USED = "Organisme_clean"

# Définir l'ordre de priorité : on prend la première valeur trouvée
cols_for_dedup = [COL_MAIN_ID, ORG_COL_USED, COL_MAIN_ADDR]

# Sécurisation si une colonne manque
cols_for_dedup = [c for c in cols_for_dedup if c in merged.columns]

df_first_occurrence = merged.dropna(subset=[COL_MAIN_ID]).drop_duplicates(subset=[COL_MAIN_ID], keep='first')[cols_for_dedup]

# Renommer les colonnes pour la fusion (Ajout de suffixe unique)
rename_map = {}
if ORG_COL_USED in df_first_occurrence.columns:
    rename_map[ORG_COL_USED] = f'{ORG_COL_USED} (Unique)'
if COL_MAIN_ADDR in df_first_occurrence.columns:
    rename_map[COL_MAIN_ADDR] = f'{COL_MAIN_ADDR} (Unique)'

df_first_occurrence = df_first_occurrence.rename(columns=rename_map)

# --- Fichier 1 : ID Aurehal avec Géolocalisation (via geonameid) DEDOUBLONNÉ ---
print("\n--- Fichier 1 : Geonames Dé-doublonné ---")

# Filtrer les lignes qui ont un geonameid
df_found_geonames = merged[merged[COL_REF_GEOID].notna()]

if not df_found_geonames.empty:
    # Ne garder qu'une seule ligne par ID Aurehal pour les infos Geonames/Ville
    # Liste des colonnes géo à garder
    cols_geo_keep = [COL_MAIN_ID, COL_REF_CITY, COL_REF_LAT, COL_REF_LON, COL_REF_GEOID, COL_INTERNAL_STATE]
    cols_geo_keep = [c for c in cols_geo_keep if c in df_found_geonames.columns]

    df_unique_geonames = df_found_geonames.drop_duplicates(subset=[COL_MAIN_ID])[cols_geo_keep]

    # Fusionner avec la première occurrence de l'organisme et de l'adresse
    df_output_geonames = df_unique_geonames.merge(
        df_first_occurrence,
        on=COL_MAIN_ID,
        how="left"
    )

    df_output_geonames.to_excel(FILE_OUTPUT_GEONAMES_DEUPL, index=False)
    print(f"🌎 Liste géolocalisée DÉDOUBLONNÉE exportée - voir si ça peut alimenter le dictionnaire : {FILE_OUTPUT_GEONAMES_DEUPL} ({len(df_output_geonames)} ID uniques)")
else:
    print(f"🚫 Aucun ID géolocalisé. Le fichier {FILE_OUTPUT_GEONAMES_DEUPL} ne sera pas créé.")

# --- Fichier 2 : ID Aurehal sans Géolocalisation (Manquants) DEDOUBLONNÉ ---
print("\n--- Fichier 2 : Manquants Dé-doublonné ---")

found_ids = merged[merged[COL_REF_GEOID].notna()][COL_MAIN_ID].unique()
df_missing_ids = merged[~merged[COL_MAIN_ID].isin(found_ids)][COL_MAIN_ID].unique()

if df_missing_ids.size > 0:
    df_output_missing = pd.DataFrame({COL_MAIN_ID: df_missing_ids})

    # Ajouter la première occurrence de l'Organisme et de l'Adresse
    df_output_missing = df_output_missing.merge(
        df_first_occurrence,
        on=COL_MAIN_ID,
        how="left"
    )

    # Ajouter l'information sur la Ville (si trouvée par crochets/recherche)
    cols_city_info = [COL_MAIN_ID, COL_REF_CITY]
    cols_city_info = [c for c in cols_city_info if c in merged.columns]
    
    df_first_city = merged.dropna(subset=[COL_MAIN_ID]).drop_duplicates(subset=[COL_MAIN_ID], keep='first')[cols_city_info]
    
    if COL_REF_CITY in df_first_city.columns:
        df_first_city = df_first_city.rename(columns={COL_REF_CITY: f'{COL_REF_CITY} (Unique)'})
    
    df_output_missing = df_output_missing.merge(
        df_first_city,
        on=COL_MAIN_ID,
        how="left"
    )
    
    df_output_missing.to_excel(FILE_OUTPUT_MISSING_DEUPL, index=False)
    print(f"📄 Liste des manquants DÉDOUBLONNÉE exportée : {FILE_OUTPUT_MISSING_DEUPL} ({len(df_output_missing)} ID uniques)")
else:
    print(f"🎉 Tous les ID ont été géolocalisés. Le fichier {FILE_OUTPUT_MISSING_DEUPL} ne sera pas créé.")

    # temps de traitement : 5mn env.

--- Chargement de cities500 ---
--- Chargement des fichiers de données ---
--- Étape 1 : Fusion par ID Aurehal ---
--- Étape 2 : Normalisation Adresses & États US ---


100%|██████████| 79530/79530 [00:00<00:00, 184205.22it/s]


-> Recherche des codes États sur 12694 lignes (Filtre USA actif)...


100%|██████████| 12694/12694 [00:00<00:00, 54873.67it/s]


--- Étape 3 : Analyse des crochets ---
--- Étape 4 : Recherche Ville dans l'adresse ---
Villes cherchées par adresse : 1540


100%|██████████| 1540/1540 [00:00<00:00, 94336.62it/s]


✅ Terminé : Copublis_Internationales_Inria_nov_2025_enrichi.xlsx

--- Fichier 1 : Geonames Dé-doublonné ---
🌎 Liste géolocalisée DÉDOUBLONNÉE exportée - voir si ça peut alimenter le dictionnaire : Copublis_Trouve_Geonames_ID_Dedoublonne.xlsx (5954 ID uniques)

--- Fichier 2 : Manquants Dé-doublonné ---
📄 Liste des manquants DÉDOUBLONNÉE exportée : Copublis_Manquants_Dedoublonne.xlsx (1560 ID uniques)


#### Comble les coordonnées manquantes de ce fichier grâce à une recherche  en cascade (Ville → Adresse → Organisme) sécurisée par une liste noire.
#### complète la recherche précédente

In [3]:
import pandas as pd
import unidecode
import re
from tqdm import tqdm

# Activer la barre de progression
tqdm.pandas()

# ---------------------------------------------------------
# 1. PARAMÈTRES LOGIQUES (Listes & Dictionnaires)
# ---------------------------------------------------------
# Mapping Pays (Indispensable pour la logique interne du moteur)
COUNTRY_MAP = {
    "united states": "US", "usa": "US", "etats-unis": "US", "us": "US", "u.s.a.": "US",
    "united kingdom": "GB", "uk": "GB", "royaume-uni": "GB", "great britain": "GB", "england": "GB", "scotland": "GB", "wales": "GB",
    "france": "FR", "germany": "DE", "allemagne": "DE", "deutschland": "DE",
    "spain": "ES", "espagne": "ES", "italy": "IT", "italie": "IT",
    "india": "IN", "inde": "IN", "china": "CN", "chine": "CN", "p.r. china": "CN",
    "canada": "CA", "australia": "AU", "brazil": "BR", "bresil": "BR",
    "netherlands": "NL", "pays-bas": "NL", "holland": "NL",
    "switzerland": "CH", "suisse": "CH", "belgium": "BE", "belgique": "BE",
    "sweden": "SE", "suede": "SE", "norway": "NO", "norvege": "NO",
    "finland": "FI", "finlande": "FI", "denmark": "DK", "danemark": "DK",
    "japan": "JP", "japon": "JP", "russia": "RU", "russie": "RU", "russian federation": "RU",
    "south korea": "KR", "coree du sud": "KR", "republic of korea": "KR",
    "austria": "AT", "autriche": "AT", "ireland": "IE", "irlande": "IE",
    "czech republic": "CZ", "czechia": "CZ", "republique tcheque": "CZ",
    "portugal": "PT", "poland": "PL", "pologne": "PL",
    "israel": "IL", "greece": "GR", "grece": "GR",
    "singapore": "SG", "singapour": "SG",
    "south africa": "ZA", "afrique du sud": "ZA",
    "argentina": "AR", "argentine": "AR", "chile": "CL", "chili": "CL",
    "colombia": "CO", "colombie": "CO", "mexico": "MX", "mexique": "MX"
}

# Liste noire (Mots à ignorer pour éviter les faux positifs)
BLOCKLIST_CITIES = {
    "argentina", "belgium", "brazil", "chile", "china", "colombia", 
    "denmark", "russia", "schweiz", "suisse", "deutschland", "france", 
    "spain", "italy", "uk", "usa", "united states", "india", "japan", "australia",
    "california", "mississippi", "texas", "florida", "massachusetts", 
    "kyushu", "okinawa", "hokkaido",
    "wales", "scotland", "england",
    "bavaria", "quebec", "ontario",
    "box", "po box", "p.o. box", "republic", "federation", "kingdom",
    "cedex", "district", "center", "centre", "campus",
    "university", "universite", "college", "institute", "school",
    "of", "the", "and", "la", "le", "de", "del"
}

# ---------------------------------------------------------
# 2. FONCTIONS DE NETTOYAGE
# ---------------------------------------------------------
def normalize_text(text):
    if pd.isna(text): return ""
    return unidecode.unidecode(str(text)).lower().strip()

def clean_text_for_search(text):
    if pd.isna(text): return ""
    txt = unidecode.unidecode(str(text)).lower()
    return re.sub(r'[^a-z0-9]', ' ', txt)

def get_country_code(country_name):
    clean = normalize_text(country_name)
    if len(clean) == 2: return clean.upper()
    return COUNTRY_MAP.get(clean, None)

def is_blocked(name):
    if name in BLOCKLIST_CITIES: return True
    if "university" in name or "institut" in name: return True
    return False

# ---------------------------------------------------------
# 3. CHARGEMENT GEONAMES
# ---------------------------------------------------------
print(f"--- 1. Chargement Geonames : {FILE_CITIES} ---")

try:
    df_geo = pd.read_csv(
        FILE_CITIES, sep="\t", header=None, quoting=3, low_memory=False, keep_default_na=False,
        usecols=[0, 1, 2, 4, 5, 8],
        names=[COL_GEO_ID, COL_GEO_NAME, COL_GEO_ASCII, COL_GEO_LAT, COL_GEO_LON, COL_GEO_COUNTRY]
    )
except NameError:
    print("❌ ERREUR CRITIQUE : Les variables de configuration ne sont pas définies.")
    raise

geo_lookup = {}          
cities_by_country = {}   
skipped_count = 0

print("   Indexation et FILTRAGE de la liste noire...")
for _, row in tqdm(df_geo.iterrows(), total=df_geo.shape[0]):
    c_code = str(row[COL_GEO_COUNTRY])
    name_local = normalize_text(row[COL_GEO_NAME])
    name_ascii = normalize_text(row[COL_GEO_ASCII])
    
    if is_blocked(name_local) or is_blocked(name_ascii):
        skipped_count += 1
        continue

    data = {'lat': row[COL_GEO_LAT], 'lon': row[COL_GEO_LON], 'id': row[COL_GEO_ID], 'name': row[COL_GEO_NAME]}
    
    if (name_ascii, c_code) not in geo_lookup:
        geo_lookup[(name_ascii, c_code)] = data
    if name_local != name_ascii:
        if (name_local, c_code) not in geo_lookup:
            geo_lookup[(name_local, c_code)] = data
    
    if c_code not in cities_by_country:
        cities_by_country[c_code] = []
    cities_by_country[c_code].append((name_ascii, data))
    if name_local != name_ascii:
        cities_by_country[c_code].append((name_local, data))

# Tri par longueur pour le scan
for c_code in cities_by_country:
    unique_cities = list({x[0]: x for x in cities_by_country[c_code]}.values())
    unique_cities.sort(key=lambda x: len(x[0]), reverse=True)
    cities_by_country[c_code] = unique_cities

# ---------------------------------------------------------
# 4. MOTEUR DE RECHERCHE CORRIGÉ
# ---------------------------------------------------------
print(f"--- 2. Traitement du fichier : {FILE_INTERIM} ---")
df_main = pd.read_excel(FILE_INTERIM)

stats = {"exist": 0, "exact": 0, "addr": 0, "org": 0, "fail": 0}
col_method = "Methode_Recherche" 

def find_location_ultimate(row):
    # 0. Si déjà présent
    if pd.notna(row.get(COL_REF_LAT)) and pd.notna(row.get(COL_REF_LON)):
        stats["exist"] += 1
        return row[COL_REF_LAT], row[COL_REF_LON], row.get(COL_REF_GEOID), row.get(COL_REF_CITY), "Déjà présent"

    # 1. Vérification Pays
    pays_raw = row.get(COL_MAIN_COUNTRY)
    country_code = get_country_code(pays_raw)
    if not country_code:
        stats["fail"] += 1
        return None, None, None, None, "Echec: Pays inconnu"

    # 2. Recherche EXACTE
    ville_raw = row.get(COL_REF_CITY)
    ville_clean = normalize_text(ville_raw)
    if ville_clean and not is_blocked(ville_clean):
        res = geo_lookup.get((ville_clean, country_code))
        if res:
            stats["exact"] += 1
            return res['lat'], res['lon'], res['id'], res['name'], "Succès: Ville Exacte"
    
    possible_cities = cities_by_country.get(country_code, [])
    
    # 3. SCAN ADRESSE
    adresse_raw = row.get(COL_MAIN_ADDR)
    if pd.notna(adresse_raw):
        addr_clean = " " + clean_text_for_search(adresse_raw) + " "
        for city_clean, data in possible_cities:
            if len(city_clean) < 3: continue 
            if f" {city_clean} " in addr_clean:
                stats["addr"] += 1
                return data['lat'], data['lon'], data['id'], data['name'], "Succès: Trouvé dans Adresse"

    # 4. SCAN ORGANISME
    org_raw = row.get(COL_MAIN_ORG)
    if pd.notna(org_raw):
        org_clean = " " + clean_text_for_search(org_raw) + " "
        for city_clean, data in possible_cities:
            if len(city_clean) < 3: continue
            if f" {city_clean} " in org_clean:
                stats["org"] += 1
                return data['lat'], data['lon'], data['id'], data['name'], "Succès: Trouvé dans Organisme"

    stats["fail"] += 1
    return None, None, None, None, "Echec: Introuvable"

print("   Lancement de la recherche...")
results = df_main.progress_apply(find_location_ultimate, axis=1, result_type='expand')


# 1. On stocke les résultats temporaires
df_main[f"{COL_REF_LAT}_New"] = results[0]
df_main[f"{COL_REF_LON}_New"] = results[1]
df_main[f"{COL_REF_GEOID}_New"] = results[2]
df_main[f"{COL_REF_CITY}_New"] = results[3]
df_main[col_method] = results[4]

# 2. On identifie ce qui est VRAIMENT vide (NaN ou chaîne de caractères vide)
mask_ville_vide = (df_main[COL_REF_CITY].isna()) | (df_main[COL_REF_CITY].astype(str).str.strip() == "")
mask_lat_vide = (df_main[COL_REF_LAT].isna())

# 3. Affectation forcée par .loc (évite les bugs du fillna sur les faux vides)
# On ne remplit que si la case d'origine était vide ET qu'on a trouvé un résultat
df_main.loc[mask_lat_vide & df_main[f"{COL_REF_LAT}_New"].notna(), COL_REF_LAT] = df_main[f"{COL_REF_LAT}_New"]
df_main.loc[mask_lat_vide & df_main[f"{COL_REF_LON}_New"].notna(), COL_REF_LON] = df_main[f"{COL_REF_LON}_New"]
df_main.loc[mask_lat_vide & df_main[f"{COL_REF_GEOID}_New"].notna(), COL_REF_GEOID] = df_main[f"{COL_REF_GEOID}_New"]
df_main.loc[mask_ville_vide & df_main[f"{COL_REF_CITY}_New"].notna(), COL_REF_CITY] = df_main[f"{COL_REF_CITY}_New"]

# Nettoyage des colonnes temporaires
df_main.drop(columns=[f"{COL_REF_LAT}_New", f"{COL_REF_LON}_New", f"{COL_REF_GEOID}_New", f"{COL_REF_CITY}_New"], inplace=True)

# ---------------------------------------------------------
# 5. EXPORT ET BILAN
# ---------------------------------------------------------
print("--- 3. Sauvegarde et Bilan ---")

df_main.to_excel(FILE_FINAL, index=False)
print(f"📄 Fichier PRINCIPAL : {FILE_FINAL}")

mask_newly_found = df_main[col_method].str.contains("Succès", na=False)
df_found = df_main[mask_newly_found]
if not df_found.empty:
    df_found.to_excel(FILE_FOUND, index=False)

mask_failed = df_main[COL_REF_LAT].isna()
df_failed = df_main[mask_failed]
if not df_failed.empty:
    df_failed.to_excel(FILE_MISSING, index=False)

# On synchronise dashboard_df pour la cellule suivante
dashboard_df = df_main.copy()

print("\n" + "="*30)
print(f"🔹 Déjà présents          : {stats['exist']}")
print(f"🔹 Trouvés (Ville Exacte) : {stats['exact']}")
print(f"🔹 Trouvés (via Adresse)  : {stats['addr']}")
print(f"🔹 Trouvés (via Organisme): {stats['org']}")
print(f"🔻 Toujours en échec      : {stats['fail']}")
print("="*30)

--- 1. Chargement Geonames : ../cities500/cities500.txt ---
   Indexation et FILTRAGE de la liste noire...


100%|██████████| 226173/226173 [00:20<00:00, 11144.51it/s]


--- 2. Traitement du fichier : Copublis_Internationales_Inria_nov_2025_enrichi.xlsx ---
   Lancement de la recherche...


100%|██████████| 79530/79530 [00:17<00:00, 4580.98it/s]


--- 3. Sauvegarde et Bilan ---
📄 Fichier PRINCIPAL : copublications_Inria_2018-2024_avec_villes.xlsx

🔹 Déjà présents          : 71660
🔹 Trouvés (Ville Exacte) : 884
🔹 Trouvés (via Adresse)  : 437
🔹 Trouvés (via Organisme): 1749
🔻 Toujours en échec      : 4800


In [4]:
import pandas as pd
import numpy as np

# On utilise dashboard_df qui est la variable de sortie de la cellule précédente
df_test = dashboard_df.copy()

print("--- 🕵️ ANALYSE DES VILLES ET COORDONNÉES ---")

# 1. Définition des masques
# Une ville est considérée comme "manquante" si elle est NaN, vide "" ou ne contient que des espaces
mask_ville_manquante = (df_test['Ville'].isna()) | (df_test['Ville'].astype(str).str.strip() == "")
mask_coord_presentes = (df_test['Latitude'].notna()) & (df_test['Longitude'].notna())

# 2. Identification des Orphelins (Coords OK mais Ville KO)
orphelins = df_test[mask_ville_manquante & mask_coord_presentes]

# 3. Identification des Échecs Totaux (Ni Ville ni Coords)
echecs_totaux = df_test[mask_ville_manquante & ~mask_coord_presentes]

# --- BILAN ---
print(f"✅ Total de lignes analysées : {len(df_test)}")
print("-" * 40)

if len(orphelins) == 0:
    print("🚀 EXCELLENT : Aucune ligne orpheline ! Toutes les coordonnées ont un nom de ville.")
else:
    print(f"⚠️ ATTENTION : Il reste {len(orphelins)} lignes avec coordonnées mais sans ville.")
    print("Détails des premières lignes problématiques :")
    # On affiche les colonnes utiles pour comprendre d'où ça vient
    cols_check = ['id_Aurehal_org_copubliant', 'Nom_org_Top_copubliant', 'Ville', 'Latitude', 'Methode_Recherche']
    cols_existantes = [c for c in cols_check if c in df_test.columns]
    display(orphelins[cols_existantes].head(10))

print("-" * 40)
print(f"ℹ️  Lignes sans aucune info (ni ville ni coords) : {len(echecs_totaux)}")
print(f"    (Celles-ci seront traitées par l'étape suivante des Capitales)")

# 4. Petit test de propreté : vérification du type de données
if not orphelins.empty:
    valeur_brute = orphelins['Ville'].iloc[0]
    print(f"\nDebug : La valeur brute de la première ville vide est : '{valeur_brute}' (Type: {type(valeur_brute)})")

--- 🕵️ ANALYSE DES VILLES ET COORDONNÉES ---
✅ Total de lignes analysées : 79530
----------------------------------------
🚀 EXCELLENT : Aucune ligne orpheline ! Toutes les coordonnées ont un nom de ville.
----------------------------------------
ℹ️  Lignes sans aucune info (ni ville ni coords) : 3887
    (Celles-ci seront traitées par l'étape suivante des Capitales)


In [5]:
#========================================================================
# Ajout de la ville correspondant à "Succès : Méthode recherche" (sauf pour US, où il manque l'Etat)
#==============================================================================
# Fonction pour extraire le nom de la ville
def extraire_ville(methode):
    if pd.isna(methode):
        return None
    match = re.search(r'\(([^)]+)\)', methode)
    if match:
        return match.group(1)
    return None

# Appliquer l'extraction uniquement si Code_Pays_orgs_copubliant != "US"
mask = df_main['Code_Pays_orgs_copubliant'] != "US"
df_main.loc[mask, 'Ville'] = df_main.loc[mask, 'Methode_Recherche'].apply(extraire_ville)

# Afficher un aperçu des résultats
# print(df_main[['Methode_Recherche', 'Code_Pays_orgs_copubliant', 'Ville']].head(10))
df_main[df_main["Hal_ID"] == "hal-01519377"][ "Ville"]


404    NaN
Name: Ville, dtype: str

In [6]:
# Liste des colonnes à conserver avec leurs nouveaux noms
dashboard_df =""
nouvelles_colonnes = {
    'Centre_inria': 'Centre',
    'Equipe_inria': 'Equipe',
    'Auteur_Inria': 'Auteurs_FR',
    'Auteur_etranger': 'Auteurs_copubliants',
    'Nom_org_copubliant': 'Organisme_copubliant',
    'id_Aurehal_org_copubliant': 'ID_Aurehal',
    'UE/Hors_UE': 'UE/Non_UE',
    'Annee': 'Année',
    'Hal_ID': 'HalID',
    'Domaine_inria': 'Domaine(s)',
    'Mots_cles_inria': 'Mots-cles',
    'Resume': 'Resume',
    'Ville': 'Ville',
    'Nom_Pays_org_copubliant': 'Pays',
    'Code_Pays_orgs_copubliant': 'Code_Pays',
    'Statecode' : 'Code_Etat',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude'
}

# Créer le nouveau DataFrame en renommant et sélectionnant les colonnes
dashboard_df = df_main.rename(columns=nouvelles_colonnes)[list(nouvelles_colonnes.values())]

# Afficher les premières lignes du nouveau DataFrame
# print(nouveau_df.head())


In [7]:
import pandas as pd

print("--- 🔍 VÉRIFICATION DU MOTEUR DE RECHERCHE ---")

# 1. On cible les lignes qui ont été trouvées par le moteur (Adresse ou Organisme)
mask_trouve = df_main['Methode_Recherche'].str.contains("Succès", na=False)
df_verif = df_main[mask_trouve]

print(f"✅ Nombre de villes trouvées par le moteur : {len(df_verif)}")

if len(df_verif) > 0:
    # 2. On compte combien parmi celles-ci n'ont pas de nom de ville
    orphelins = df_verif[df_verif[COL_REF_CITY].isna() | (df_verif[COL_REF_CITY] == "")]
    
    if len(orphelins) == 0:
        print("🚀 EXCELLENT : Toutes les villes trouvées ont désormais un nom ET des coordonnées.")
    else:
        print(f"⚠️ ATTENTION : Il reste encore {len(orphelins)} lignes orphelines.")
        print("Voici un échantillon des lignes à problème :")
        display(orphelins[[COL_MAIN_ADDR, COL_REF_CITY, COL_REF_LAT, 'Methode_Recherche']].head())

# 3. Test de cohérence globale
total_orphelins_global = (
    (df_main[COL_REF_LAT].notna()) & 
    (df_main[COL_REF_CITY].isna() | (df_main[COL_REF_CITY] == ""))
).sum()

print(f"\n📊 BILAN GLOBAL :")
print(f"📍 Nombre total de lignes avec Coordonnées mais SANS Ville : {total_orphelins_global}")

if total_orphelins_global == 0:
    print("🏆 Bravo ! Votre base est parfaitement cohérente. Vous pouvez passer à la suite.")
else:
    print("👀 Il reste des orphelins. Ils viennent probablement de la fusion Aurehal initiale (Etape 1).")

--- 🔍 VÉRIFICATION DU MOTEUR DE RECHERCHE ---
✅ Nombre de villes trouvées par le moteur : 3070
⚠️ ATTENTION : Il reste encore 2560 lignes orphelines.
Voici un échantillon des lignes à problème :


,Adresse_org_Top_copubliant,Ville,Latitude,Methode_Recherche
18,"Gower Street, London WC1E 6BT",NaN,51.12472,Succès: Trouvé dans Adresse
27,"Strand Campus, London WC2R 2LS",NaN,51.50853,Succès: Ville Exacte
404,NaN,NaN,51.05089,Succès: Trouvé dans Organisme
724,"Mile End Road, London E1 4NS",NaN,51.50853,Succès: Ville Exacte
726,"Mile End Road, London E1 4NS",NaN,51.50853,Succès: Ville Exacte



📊 BILAN GLOBAL :
📍 Nombre total de lignes avec Coordonnées mais SANS Ville : 60792
👀 Il reste des orphelins. Ils viennent probablement de la fusion Aurehal initiale (Etape 1).


In [8]:

dashboard_df[dashboard_df["Code_Pays"] == "US"]

,Centre,Equipe,Auteurs_FR,Auteurs_copubliants,Organisme_copubliant,ID_Aurehal,UE/Non_UE,Année,HalID,Domaine(s),Mots-cles,Resume,Ville,Pays,Code_Pays,Code_Etat,Latitude,Longitude
1,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Samaranayake, Samitha",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
2,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Reilly, Jack",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
3,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Krichene, Walid",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
4,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Bayen, Alexandre",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
5,Inria Univ. Cote Azur,ACUMES,"Goatin, Paola","Samaranayake, Samitha",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","EDP instationnaires, contrôle optimal, quantif...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79442,Inria Paris,MOKAPLAN,"Léger, Flavien","Brendle, Simon",Columbia University [New York],75524,Hors_UE,2024,hal-05474001,"Mathématiques appliquées, calcul et simulation",NaN,A key inequality which underpins the regularit...,New York City,United States,US,NY,40.71427,-74.00597
79512,Inria Paris,SERENA,"Vohralík, Martin","Yotov, Ivan",Department of Mathematics,110124,Hors_UE,2024,hal-04611932,"Santé, biologie et planète numériques","écoulements multiphasiques de Darcy, écoulemen...","In this work, we develop algebraic solvers for...",Pittsburgh,United States,US,PA,40.44062,-79.99589
79513,Inria Univ. Lorraine,CARAMBA,"Zimmermann, Paul","Caprioli, Paul",High Performance Kernels LLC,1209204,Hors_UE,2024,hal-04714173,"Algorithmique, programmation, logiciels et arc...","Cryptographie, Factorisation d'entiers, Logari...",NaN,NaN,United States,US,NaN,NaN,NaN
79526,Inria Saclay UPS,AVIZ,"Yao, Lijie","Lin, Tica",Harvard University,38302,Hors_UE,2024,hal-05519489,"Perception, Cognition, Interaction","Visualisation de données, Interaction Homme-Ma...",This half-day workshop will gather researchers...,Cambridge,United States,US,MA,42.37510,-71.10561


#### Nettoyage et complétion du dataframe pour export vers dashboard

In [9]:
#=========================================================
#  Suppression des tirets en début de texte
#=========================================================

colonnes_a_exclure = ['Latitude', 'Longitude','HalID']


# Fonction pour retirer les préfixes comme " - " ou " -long" au début des valeurs
def nettoyer_valeur(valeur):
    if pd.isna(valeur):
        return valeur
    # Retirer tout préfixe commençant par " - " suivi de n'importe quel texte
    return re.sub(r'^\s*-\s*', '', str(valeur)).strip()

# Appliquer la fonction à toutes les colonnes de type 'object' (chaînes de caractères)
for colonne in dashboard_df.select_dtypes(include='object').columns:
    if colonne not in colonnes_a_exclure:
        dashboard_df[colonne] = dashboard_df[colonne].apply(nettoyer_valeur)

C:\Users\dadasilv\AppData\Local\Temp\ipykernel_21792\1198068966.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for colonne in dashboard_df.select_dtypes(include='object').columns:


In [10]:
dashboard_df[dashboard_df["Code_Pays"] == "US"]

,Centre,Equipe,Auteurs_FR,Auteurs_copubliants,Organisme_copubliant,ID_Aurehal,UE/Non_UE,Année,HalID,Domaine(s),Mots-cles,Resume,Ville,Pays,Code_Pays,Code_Etat,Latitude,Longitude
1,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Samaranayake, Samitha",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
2,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Reilly, Jack",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
3,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Krichene, Walid",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
4,Inria Univ. Grenoble,NECS,"Delle Monache, Maria Laura","Bayen, Alexandre",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","commande de systèmes complexes, réseau de capt...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
5,Inria Univ. Cote Azur,ACUMES,"Goatin, Paola","Samaranayake, Samitha",Department of Civil and Environmental Engineer...,133227,Hors_UE,2018,hal-01095707,"Mathématiques appliquées, calcul et simulation","EDP instationnaires, contrôle optimal, quantif...",We consider the System Optimal Dynamic Traffic...,Berkeley,United States,US,CA,37.87159,-122.27275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79442,Inria Paris,MOKAPLAN,"Léger, Flavien","Brendle, Simon",Columbia University [New York],75524,Hors_UE,2024,hal-05474001,"Mathématiques appliquées, calcul et simulation",NaN,A key inequality which underpins the regularit...,New York City,United States,US,NY,40.71427,-74.00597
79512,Inria Paris,SERENA,"Vohralík, Martin","Yotov, Ivan",Department of Mathematics,110124,Hors_UE,2024,hal-04611932,"Santé, biologie et planète numériques","écoulements multiphasiques de Darcy, écoulemen...","In this work, we develop algebraic solvers for...",Pittsburgh,United States,US,PA,40.44062,-79.99589
79513,Inria Univ. Lorraine,CARAMBA,"Zimmermann, Paul","Caprioli, Paul",High Performance Kernels LLC,1209204,Hors_UE,2024,hal-04714173,"Algorithmique, programmation, logiciels et arc...","Cryptographie, Factorisation d'entiers, Logari...",NaN,NaN,United States,US,NaN,NaN,NaN
79526,Inria Saclay UPS,AVIZ,"Yao, Lijie","Lin, Tica",Harvard University,38302,Hors_UE,2024,hal-05519489,"Perception, Cognition, Interaction","Visualisation de données, Interaction Homme-Ma...",This half-day workshop will gather researchers...,Cambridge,United States,US,MA,42.37510,-71.10561


In [11]:
import pandas as pd

# =============================================================================
# ANALYSE DES INCOHÉRENCES DE COORDONNÉES AVANT STANDARDISATION
# =============================================================================

print("--- 🕵️‍♂️ Recherche des incohérences de coordonnées ---")

# 1. On travaille sur une copie pour ne pas toucher aux données
df_check = dashboard_df.copy()

# 2. On crée une colonne "Couple" (Lat, Lon) pour faciliter la comparaison
# On convertit en string pour que ce soit hashable et lisible
df_check['coord_couple'] = df_check.apply(
    lambda row: f"({row['Latitude']}, {row['Longitude']})", axis=1
)

# 3. On groupe par Ville + Pays + Etat (pour les US) et on compte le nombre de couples UNIQUES
# On remplace les NaN dans l'état par "" pour éviter les soucis de groupement
df_check['Code_Etat'] = df_check['Code_Etat'].fillna("")

groupe_ville = df_check.groupby(['Ville', 'Code_Pays', 'Code_Etat'])['coord_couple'].nunique()

# 4. On filtre pour ne garder que ceux qui ont > 1 version de coordonnées
villes_incoherentes = groupe_ville[groupe_ville > 1]

print(f"⚠️ Nombre de villes ayant des coordonnées multiples (flou géographique) : {len(villes_incoherentes)}")
print("="*60)

# 5. Affichage détaillé des 10 pires cas (ou les premiers trouvés)
if len(villes_incoherentes) > 0:
    count = 0
    # On trie pour voir les villes avec le plus de variantes en premier (optionnel)
    for (ville, pays, etat), nb_variantes in villes_incoherentes.sort_values(ascending=False).items():
        if count >= 10: break # On affiche juste les 10 premiers exemples
        
        print(f"\n📍 VILLE : {ville} (Pays: {pays}, État: {etat})")
        print(f"   -> Possède {nb_variantes} localisations différentes dans le fichier :")
        
        # On récupère les détails pour cette ville
        mask = (df_check['Ville'] == ville) & (df_check['Code_Pays'] == pays) & (df_check['Code_Etat'] == etat)
        details = df_check[mask]['coord_couple'].value_counts()
        
        for coord, qte in details.items():
            print(f"      🔸 Coordonnées {coord} : présent sur {qte} lignes")
            
        count += 1
else:
    print("✅ Aucune incohérence détectée ! Toutes les villes ont des coordonnées uniques.")

print("\n" + "="*60)
print("💡 Ces écarts seront corrigés par la cellule suivante (Standardisation).")

--- 🕵️‍♂️ Recherche des incohérences de coordonnées ---
⚠️ Nombre de villes ayant des coordonnées multiples (flou géographique) : 10

📍 VILLE : Berkeley (Pays: US, État: CA)
   -> Possède 2 localisations différentes dans le fichier :
      🔸 Coordonnées (37.87159, -122.27275) : présent sur 447 lignes
      🔸 Coordonnées (38.7545, -90.33123) : présent sur 34 lignes

📍 VILLE : California (Pays: US, État: CA)
   -> Possède 2 localisations différentes dans le fichier :
      🔸 Coordonnées (42.03471, -93.61994) : présent sur 4 lignes
      🔸 Coordonnées (nan, nan) : présent sur 1 lignes

📍 VILLE : Charlestown (Pays: US, État: MA)
   -> Possède 2 localisations différentes dans le fichier :
      🔸 Coordonnées (42.37787, -71.062) : présent sur 28 lignes
      🔸 Coordonnées (38.45312, -85.67024) : présent sur 2 lignes

📍 VILLE : Mountain View (Pays: US, État: CA)
   -> Possède 2 localisations différentes dans le fichier :
      🔸 Coordonnées (37.38605, -122.08385) : présent sur 27 lignes
     

#### Ajout des capitales + lat et long pour les villes non trouvées

In [12]:
#=====================================================
# Ajout des capitales + lat et long pour les villes non trouvées
#=====================================================

# Charger les fichiers GeoNames
def charger_donnees_geonames():
    # Charger countryInfo.txt
    country_df = pd.read_csv('countryInfo.txt', sep='\t', comment='#', names=[
        'ISO', 'ISO3', 'ISO-Numeric', 'fips', 'Country', 'Capital',
        'Area', 'Population', 'Continent', 'tld', 'CurrencyCode', 'CurrencyName',
        'Phone', 'Postal Code Format', 'Postal Code Regex', 'Languages',
        'geonameid', 'neighbours', 'EquivalentFipsCode'
    ])

    # Charger cities500.txt
    cities_df = pd.read_csv('../cities500/cities500.txt', sep='\t', comment='#', names=[
        'geonameid', 'name', 'asciiname', 'alternatenames', 'latitude', 'longitude',
        'feature class', 'feature code', 'country code', 'cc2', 'admin1 code',
        'admin2 code', 'admin3 code', 'admin4 code', 'population', 'elevation',
        'dem', 'timezone', 'modification date'
    ])

    return country_df, cities_df

# Fonction pour obtenir la capitale d'un pays
def obtenir_capitale(code_pays, country_df):
    capital_row = country_df[country_df['ISO'] == code_pays]
    if not capital_row.empty:
        return capital_row.iloc[0]['Capital']
    return None

# Fonction pour obtenir les coordonnées d'une ville
def obtenir_coordonnees(ville, code_pays, cities_df):
    ville_row = cities_df[(cities_df['name'] == ville) & (cities_df['country code'] == code_pays)]
    if not ville_row.empty:
        return ville_row.iloc[0]['latitude'], ville_row.iloc[0]['longitude']
    return None, None

# Charger les données GeoNames
country_df, cities_df = charger_donnees_geonames()

# Extraire les valeurs uniques de la colonne 'Code_Pays'
pays_uniques = dashboard_df['Code_Pays'].unique()

# Créer un dictionnaire des pays et de leurs capitales
capitales = {}
coordonnees = {}

for code_pays in pays_uniques:
    if pd.notna(code_pays) and code_pays != "":
        capitale = obtenir_capitale(code_pays, country_df)
        # print(f"Pays: {code_pays}, Capitale: {capitale}")
        capitales[code_pays] = capitale
        if capitale:
            latitude, longitude = obtenir_coordonnees(capitale, code_pays, cities_df)
            coordonnees[code_pays] = (latitude, longitude)
            # print(f"Coordonnées de {capitale}: Latitude {latitude}, Longitude {longitude}")

# Créer un masque pour les valeurs vides dans 'Ville'
mask_villes_vides = dashboard_df['Ville'].isin([""]) | dashboard_df['Ville'].isna()

# Remplir les valeurs vides dans 'Ville' avec la capitale du pays
dashboard_df.loc[mask_villes_vides, 'Ville'] = dashboard_df.loc[mask_villes_vides, 'Code_Pays'].map(capitales)

# Créer un masque pour les valeurs vides dans 'Latitude' et 'Longitude'
mask_coord_vides = (dashboard_df['Latitude'].isin([""])) | dashboard_df['Latitude'].isna()

# Remplir les valeurs de latitude et longitude uniquement si elles sont vides
for code_pays, (latitude, longitude) in coordonnees.items():
    mask = (dashboard_df['Code_Pays'] == code_pays) & mask_coord_vides & (dashboard_df['Ville'] == capitales[code_pays])
    dashboard_df.loc[mask, 'Latitude'] = latitude
    dashboard_df.loc[mask, 'Longitude'] = longitude

# Filtrer les lignes où la colonne 'Ville' a été remplie avec une capitale
villes_ajoutees = dashboard_df.loc[mask_villes_vides, 'Ville'].dropna()

# Obtenir la liste unique des villes ajoutées
liste_villes_uniques = villes_ajoutees.nunique()

# Obtenir le nombre total de villes renseignées
nombre_villes_renseignees = len(villes_ajoutees)

# Afficher les résultats
print("Liste unique des villes ajoutées :")
print(liste_villes_uniques)


# Temps de traitement : 6 à 10 sec

C:\Users\dadasilv\AppData\Local\Temp\ipykernel_21792\3499397598.py:16: DtypeWarning: Columns (0: admin3 code, 1: admin4 code) have mixed types. Specify dtype option on import or set low_memory=False.
  cities_df = pd.read_csv('../cities500/cities500.txt', sep='\t', comment='#', names=[


Liste unique des villes ajoutées :
120


In [13]:
import pandas as pd

# =============================================================================
# ANALYSE DES INCOHÉRENCES DE COORDONNÉES AVANT STANDARDISATION
# =============================================================================

print("--- 🕵️‍♂️ Recherche des incohérences de coordonnées ---")

# 1. On travaille sur une copie pour ne pas toucher aux données
df_check = dashboard_df.copy()

# 2. On crée une colonne "Couple" (Lat, Lon) pour faciliter la comparaison
# On convertit en string pour que ce soit hashable et lisible
df_check['coord_couple'] = df_check.apply(
    lambda row: f"({row['Latitude']}, {row['Longitude']})", axis=1
)

# 3. On groupe par Ville + Pays + Etat (pour les US) et on compte le nombre de couples UNIQUES
# On remplace les NaN dans l'état par "" pour éviter les soucis de groupement
df_check['Code_Etat'] = df_check['Code_Etat'].fillna("")

groupe_ville = df_check.groupby(['Ville', 'Code_Pays', 'Code_Etat'])['coord_couple'].nunique()

# 4. On filtre pour ne garder que ceux qui ont > 1 version de coordonnées
villes_incoherentes = groupe_ville[groupe_ville > 1]

print(f"⚠️ Nombre de villes ayant des coordonnées multiples (flou géographique) : {len(villes_incoherentes)}")
print("="*60)

# 5. Affichage détaillé des 10 pires cas (ou les premiers trouvés)
if len(villes_incoherentes) > 0:
    count = 0
    # On trie pour voir les villes avec le plus de variantes en premier (optionnel)
    for (ville, pays, etat), nb_variantes in villes_incoherentes.sort_values(ascending=False).items():
        if count >= 10: break # On affiche juste les 10 premiers exemples
        
        print(f"\n📍 VILLE : {ville} (Pays: {pays}, État: {etat})")
        print(f"   -> Possède {nb_variantes} localisations différentes dans le fichier :")
        
        # On récupère les détails pour cette ville
        mask = (df_check['Ville'] == ville) & (df_check['Code_Pays'] == pays) & (df_check['Code_Etat'] == etat)
        details = df_check[mask]['coord_couple'].value_counts()
        
        for coord, qte in details.items():
            print(f"      🔸 Coordonnées {coord} : présent sur {qte} lignes")
            
        count += 1
else:
    print("✅ Aucune incohérence détectée ! Toutes les villes ont des coordonnées uniques.")

print("\n" + "="*60)
print("💡 Ces écarts seront corrigés par la cellule suivante (Standardisation).")

--- 🕵️‍♂️ Recherche des incohérences de coordonnées ---
⚠️ Nombre de villes ayant des coordonnées multiples (flou géographique) : 90

📍 VILLE : Berlin (Pays: DE, État: )
   -> Possède 126 localisations différentes dans le fichier :
      🔸 Coordonnées (52.52437, 13.41053) : présent sur 1210 lignes
      🔸 Coordonnées (49.23262, 7.00982) : présent sur 603 lignes
      🔸 Coordonnées (48.13743, 11.57549) : présent sur 531 lignes
      🔸 Coordonnées (49.40768, 8.69079) : présent sur 478 lignes
      🔸 Coordonnées (51.48165, 7.21648) : présent sur 318 lignes
      🔸 Coordonnées (50.77664, 6.08342) : présent sur 304 lignes
      🔸 Coordonnées (48.24896, 11.65101) : présent sur 275 lignes
      🔸 Coordonnées (50.92149, 6.36267) : présent sur 264 lignes
      🔸 Coordonnées (51.05089, 13.73832) : présent sur 241 lignes
      🔸 Coordonnées (48.78232, 9.17702) : présent sur 226 lignes
      🔸 Coordonnées (49.00937, 8.40444) : présent sur 222 lignes
      🔸 Coordonnées (51.53443, 9.93228) : présen

#### STANDARDISATION DES COORDONNÉES

In [14]:
# =============================================================================
# ÉTAPE DE NORMALISATION / STANDARDISATION DES COORDONNÉES
# =============================================================================
# Objectif : S'assurer que chaque couple (Ville, Pays) possède une Latitude/Longitude UNIQUE.
# On écrase les coordonnées existantes (issues de sources variées) par celles
# du référentiel officiel Geonames (cities500) pour éviter les doublons flous sur la carte.
# =============================================================================

import pandas as pd
from tqdm import tqdm

print("--- 🔄 Démarrage de la standardisation des coordonnées ---")

# 1. Chargement propre du référentiel Geonames
# On ne charge que les colonnes utiles pour économiser la mémoire
print("   Chargement du référentiel Geonames...")
df_geo_ref = pd.read_csv(
    '../cities500/cities500.txt',
    sep='\t', comment='#',
    usecols=[1, 4, 5, 8, 10],  # name, lat, lon, country code, admin1 code
    names=['name', 'latitude', 'longitude', 'country_code', 'admin1_code'],
    dtype=str  # On force le string pour le matching
)

# Conversion des lat/lon en float pour le résultat final
df_geo_ref['latitude'] = pd.to_numeric(df_geo_ref['latitude'], errors='coerce')
df_geo_ref['longitude'] = pd.to_numeric(df_geo_ref['longitude'], errors='coerce')

# 2. Création des Dictionnaires de recherche (Hashmaps)
# Cela permet une recherche instantanée sans scanner tout le tableau à chaque ligne
# On stocke les clés en MINUSCULE pour rendre la recherche insensible à la casse
dict_geo_simple = {}  # Clé : (ville, pays)
dict_geo_state = {}   # Clé : (ville, pays, etat)

print("   Indexation du référentiel...")
for row in tqdm(df_geo_ref.itertuples(index=False), total=len(df_geo_ref)):
    # Normalisation des textes pour la clé (minuscule, sans espace autour)
    nom_ville = str(row.name).strip().lower()
    code_pays = str(row.country_code).strip().upper()
    
    coords = (row.latitude, row.longitude)
    
    # Remplissage Dict Simple (Ville + Pays)
    # Note : S'il y a des doublons dans Geonames, le dernier lu écrase (souvent la ville la plus importante)
    dict_geo_simple[(nom_ville, code_pays)] = coords
    
    # Remplissage Dict avec Etat (Ville + Pays + Code Etat)
    if pd.notna(row.admin1_code):
        code_etat = str(row.admin1_code).strip().upper()
        dict_geo_state[(nom_ville, code_pays, code_etat)] = coords

# 3. Fonction d'application (Logique de remplacement)
def get_standardized_coords(row):
    # Récupération des données de la ligne dashboard
    ville_in = str(row['Ville']).strip().lower() if pd.notna(row['Ville']) else ""
    pays_in = str(row['Code_Pays']).strip().upper() if pd.notna(row['Code_Pays']) else ""
    etat_in = str(row['Code_Etat']).strip().upper() if pd.notna(row['Code_Etat']) else ""
    
    # Sécurité : Si pas de ville ou pas de pays, on garde l'existant
    if not ville_in or not pays_in:
        return row['Latitude'], row['Longitude']

    # ESSAI 1 : Recherche précise avec l'État (Prioritaire pour US, CA, etc.)
    # On ne tente ça que si on a un code état dans le dashboard
    if etat_in:
        res = dict_geo_state.get((ville_in, pays_in, etat_in))
        if res:
            return res # On renvoie les coords standardisées

    # ESSAI 2 : Recherche Ville + Pays (Standard)
    res = dict_geo_simple.get((ville_in, pays_in))
    if res:
        return res
        
    # FALLBACK : Si la ville n'est pas dans Geonames (ex: petite ville introuvable),
    # on garde les coordonnées qu'on avait déjà (calculées précédemment ou Aurehal)
    return row['Latitude'], row['Longitude']

# 4. Exécution sur le DataFrame Dashboard
print("   Application de la standardisation sur les données...")
# tqdm.pandas() # Si tqdm est activé pour pandas
dashboard_df[['Latitude', 'Longitude']] = dashboard_df.progress_apply(get_standardized_coords, axis=1, result_type='expand')

print("✅ Standardisation terminée. Cohérence géographique assurée.")

--- 🔄 Démarrage de la standardisation des coordonnées ---
   Chargement du référentiel Geonames...
   Indexation du référentiel...


100%|██████████| 226173/226173 [00:01<00:00, 127198.56it/s]


   Application de la standardisation sur les données...


100%|██████████| 79530/79530 [00:03<00:00, 20895.80it/s]

✅ Standardisation terminée. Cohérence géographique assurée.


In [15]:
#===================================================================================
# Suppression de tous les caractères non latins, normalisation des caractères accentués
#=================================================================================
import pandas as pd
import re
from unidecode import unidecode  # pip install unidecode si nécessaire

# =========================================================
# 1. Colonnes texte
# =========================================================
colonnes_texte = dashboard_df.select_dtypes(include='object').columns

# =========================================================
# 2. Pattern des caractères à supprimer
#    arabe, cyrillique, chinois, japonais, symboles - et /
# =========================================================
pattern_exotiques = re.compile(r'[\u0600-\u06FF\u0400-\u04FF\u3040-\u30FF\u4E00-\u9FFF\-/]')

# =========================================================
# 3. Fonction de nettoyage et normalisation
# =========================================================
def normaliser_et_nettoyer(texte):
    if pd.isna(texte):
        return texte
    # 1️⃣ Supprimer les caractères exotiques et symboles
    texte_nettoye = pattern_exotiques.sub('', str(texte))
    # 2️⃣ Normaliser les lettres accentuées en ASCII
    texte_normalise = unidecode(texte_nettoye)
    # 3️⃣ Nettoyage optionnel : retirer espaces multiples au passage
    texte_normalise = re.sub(r'\s+', ' ', texte_normalise).strip()
    return texte_normalise

# =========================================================
# 4. Application à toutes les colonnes texte
# =========================================================
for col in colonnes_texte:
    dashboard_df[col] = dashboard_df[col].apply(normaliser_et_nettoyer)

print("Terminé")

# Temps de traitement : env. 30 sec.

C:\Users\dadasilv\AppData\Local\Temp\ipykernel_21792\3138743469.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = dashboard_df.select_dtypes(include='object').columns


Terminé


In [16]:
#==================================
# Vérification de la présence de caractères pouvant gêner le dashboard
#========================================================================

# =========================================================
# 1. Définition des patterns Unicode interdits
# =========================================================
patterns = {
    'arabe': re.compile(r'[\u0600-\u06FF]'),
    'cyrillique': re.compile(r'[\u0400-\u04FF]'),
    'chinois': re.compile(r'[\u4E00-\u9FFF]'),
    'japonais': re.compile(r'[\u3040-\u30FF]'),  # hiragana + katakana
    'polonais': re.compile(r'[ąćęłńóśżźĄĆĘŁŃÓŚŻŹ]')
}

# =========================================================
# 2. Sélection des colonnes texte uniquement
# =========================================================
colonnes_texte = dashboard_df.select_dtypes(include='object').columns

# =========================================================
# 3. Recherche des caractères interdits
# =========================================================
resultats = []

for col in colonnes_texte:
    for idx, valeur in dashboard_df[col].dropna().items():
        texte = str(valeur)

        for langue, pattern in patterns.items():
            if pattern.search(texte):
                resultats.append({
                    'index': idx,
                    'colonne': col,
                    'langue_detectee': langue,
                    'valeur': texte
                })

# =========================================================
# 4. Résultat final
# =========================================================
resultats_df = pd.DataFrame(resultats)

if resultats_df.empty:
    print("✅ Aucun caractère arabe, cyrillique, chinois, japonais, polonais détecté.")
else:
    print("⚠️ Caractères non autorisés détectés :")
    print(resultats_df)


C:\Users\dadasilv\AppData\Local\Temp\ipykernel_21792\3956098721.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = dashboard_df.select_dtypes(include='object').columns


✅ Aucun caractère arabe, cyrillique, chinois, japonais, polonais détecté.


In [17]:
import pandas as pd
import re

# =========================================================
# 
# Suppression des balises  et codes html dans la colonne Resume
# =========================================================
def nettoyer_resume(texte):
    if pd.isna(texte):
        return texte
    
    texte = str(texte)
    
    # 1️⃣ Supprimer toutes les balises HTML <...>
    texte = re.sub(r'<.*?>', '', texte)
    
    # 2️⃣ Supprimer tous les codes HTML du type &acute; &nbsp; &amp; etc.
    texte = re.sub(r'&\w+;', '', texte)
    
    # 3️⃣ Garder tout à partir de la première lettre majuscule
    match = re.search(r'[A-Z]', texte)
    if match:
        texte = texte[match.start():]
    
    # 4️⃣ Nettoyage final des espaces en début/fin
    return texte.strip()

# =========================================================
# 2. Application sur la colonne Resume
# =========================================================
dashboard_df['Resume'] = dashboard_df['Resume'].apply(nettoyer_resume)

print("Nettoyage résumé terminé")


Nettoyage résumé terminé


In [18]:
dashboard_df = dashboard_df.fillna("")

dashboard_df.to_excel("dashboard.xlsx", index=False)


In [19]:
dashboard_df = dashboard_df.fillna("")

dashboard_df.to_csv("Copublis_dashboard.csv")

In [20]:
nombre_hal_id_uniques = dashboard_df['HalID'].nunique()
print(f"nombre total de publications:{nombre_hal_id_uniques}")


nombre total de publications:14354


In [21]:
import pandas as pd

# Nom de votre fichier Aurehal
FILE_REF = "ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx"

print(f"--- 🕵️‍♂️ INSPECTION CIBLÉE DANS : {FILE_REF} ---")

try:
    # On charge le fichier sans convertir les "None" ou "NaN" automatiquement
    # pour voir ce qui est écrit TEXTUELLEMENT.
    df = pd.read_excel(FILE_REF, keep_default_na=False, na_values=[''])
    
    print("✅ Fichier chargé.")
    
    # On cherche les coordonnées suspectes (arrondies pour être sûr de trouver)
    # Latitude env 44.93 et Longitude env 7.54
    # On convertit en numérique pour éviter les erreurs de format texte
    df['Latitude'] = pd.to_numeric(df['Latitude'], errors='coerce')
    df['Longitude'] = pd.to_numeric(df['Longitude'], errors='coerce')

    # Masque pour trouver cette ligne précise
    mask_suspect = (df['Latitude'] > 44.9) & (df['Latitude'] < 45.0) & \
                   (df['Longitude'] > 7.5) & (df['Longitude'] < 7.6)
    
    lignes_trouvees = df[mask_suspect]
    
    print(f"\n🔎 Recherche des coordonnées 'Turin/Rome' (44.93, 7.54) :")
    
    if len(lignes_trouvees) > 0:
        print(f"   👉 TROUVÉ ! Il y a {len(lignes_trouvees)} ligne(s) correspondante(s).")
        print("\n   Voici le contenu BRUT de la colonne 'Ville' :")
        
        for index, row in lignes_trouvees.iterrows():
            ville_raw = row['Ville']
            pays_raw = row['Nom_Pays_org_copubliant']
            id_aurehal = row['id_Aurehal_org_copubliant']
            
            print(f"   -------------------------------------------------")
            print(f"   🆔 ID Aurehal : {id_aurehal}")
            print(f"   🌍 Pays       : {pays_raw}")
            print(f"   📍 Latitude   : {row['Latitude']}")
            print(f"   🏙️ VILLE (Brut): '{ville_raw}'")
            print(f"   ❓ Est-ce vide ? : {ville_raw == ''}")
            print(f"   ❓ Est-ce 'None'? : {str(ville_raw).lower() == 'none'}")
            
            if str(ville_raw).lower() == 'none':
                print("   🚨 COUPABLE IDENTIFIÉ : La ville s'appelle littéralement 'None'.")
                print("      Pandas le lit comme un vide si on ne fait pas attention !")
    else:
        print("   ❌ Ces coordonnées ne sont pas dans le fichier Aurehal.")

except FileNotFoundError:
    print(f"❌ Impossible de trouver le fichier : {FILE_REF}")
except Exception as e:
    print(f"❌ Erreur : {e}")

--- 🕵️‍♂️ INSPECTION CIBLÉE DANS : ID_Aurehal_Ville_Etat_Latitude_Longitude.xlsx ---
✅ Fichier chargé.

🔎 Recherche des coordonnées 'Turin/Rome' (44.93, 7.54) :
   ❌ Ces coordonnées ne sont pas dans le fichier Aurehal.
